1. 中间文件处理

In [1]:
import pandas as pd

# 读取原始csv文件
df = pd.read_csv('/home/mw/input/data_total74037403/行业角度数据集/上市公司词频.csv', encoding='utf-8')

# 提取需要的列
cols = ['stkcd', 'year', '证券简称', '年报全文人工智能关键词词频', '年报全文人工智能关键词词频加1取自然对数']
df_simple = df[cols]

# 保存为新csv文件
df_simple.to_csv('词频数据简化.csv', index=False, encoding='utf-8-sig')


In [2]:
import pandas as pd

# 读取数据
df = pd.read_csv('/home/mw/input/data_total74037403/行业角度数据集/词频数据简化.csv')

# 找到每个stkcd最新年份的证券简称
latest_names = df.sort_values('year').groupby('stkcd').last()['证券简称']

# 用最新证券简称替换所有年份
df['证券简称'] = df['stkcd'].map(latest_names)

# 保存结果
df.to_csv('词频数据简化_统一简称.csv', index=False)

In [3]:
import pandas as pd

# 1. 读取企业规模对照表
scale_df = pd.read_csv('/home/mw/input/data_total74037403/行业角度数据集/词频_企业规模.csv')  

scale_dict = dict(zip(scale_df['证券简称'], scale_df['企业规模']))

# 2. 读取专利数据
patent_df = pd.read_csv('/home/mw/input/data_total74037403/行业角度数据集/词频数据简化_统一简称.csv', encoding='utf-8')

# 3. 添加企业规模列
patent_df['企业规模'] = patent_df['证券简称'].map(scale_dict)

# 4. 保存新文件
patent_df.to_csv('词频数据最终文件.csv', index=False, encoding='utf-8-sig')

2. 可视化

In [4]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
# 设置支持中文的字体
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']  
plt.rcParams['axes.unicode_minus'] = False 

In [5]:
import pandas as pd

# 读取数据
df = pd.read_csv('/home/mw/input/data_total74037403/行业角度数据集/词频数据最终文件.csv')

# 计算2016年大型企业的平均年报全文人工智能关键词词频
df_2016_large = df[(df['year'] == 2016) & (df['企业规模'] == '大型企业')]

# 计算平均值
mean_ai_freq = df_2016_large['年报全文人工智能关键词词频'].mean()

print(f"2016年大型企业的平均年报全文人工智能关键词词频为：{mean_ai_freq}")

2016年大型企业的平均年报全文人工智能关键词词频为：3.862521391899601


In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle

# 读取数据
df = pd.read_csv('/home/mw/input/data_total74037403/行业角度数据集/词频数据最终文件.csv')

# 分别过滤三类企业
df_large = df[df['企业规模'] == '大型企业']
df_mid = df[df['企业规模'] == '中型企业']
df_small = df[df['企业规模'] == '小型企业']

# 按年份分组计算平均值
mean_large = df_large.groupby('year')['年报全文人工智能关键词词频'].mean().loc[2016:2023]
mean_mid = df_mid.groupby('year')['年报全文人工智能关键词词频'].mean().loc[2016:2023]
mean_small = df_small.groupby('year')['年报全文人工智能关键词词频'].mean().loc[2016:2023]

plt.figure(figsize=(10,6))
plt.plot(mean_large.index, mean_large.values, marker='o', label='大型企业', linewidth=2.5, color="#4F77CE")
plt.plot(mean_mid.index, mean_mid.values, marker='s', label='中型企业', linewidth=2.5, color="#F09935")

bar_width = 0.35
years = mean_small.index.astype(int)

# 渐变色设置（浅橙到深橙）
color_top = np.array([0.32156863, 0.57254902, 0.4])  
color_bottom = np.array([0.65882353, 0.74901961, 0.68627451]) 

for i, (x, y) in enumerate(zip(years, mean_small.values)):
    n = 100  # 渐变分层数
    for j in range(n):
        c = color_bottom * (1 - j/n) + color_top * (j/n)
        plt.gca().add_patch(Rectangle(
            (x + bar_width/2, y*j/n), bar_width, y/n,
            color=c, linewidth=0, zorder=1
        ))
# 添加柱形图例
plt.bar(years + bar_width/2, mean_small.values, width=bar_width, color=color_top, alpha=0.0, label='小型企业')

# 在柱形和折线旁边显示数据
for x, y in zip(years, mean_small.values):
    plt.text(x + bar_width/2, y, f'{y:.1f}', ha='center', va='bottom', fontsize=10, color="#93CBA4")
for x, y in zip(mean_large.index, mean_large.values):
    plt.text(x, y, f'{y:.1f}', ha='center', va='bottom', fontsize=10, color="#1f77b4")
for x, y in zip(mean_mid.index, mean_mid.values):
    plt.text(x, y, f'{y:.1f}', ha='center', va='bottom', fontsize=10, color="#e5a225")

plt.title('2016-2023年不同规模企业平均年报全文人工智能关键词词频')
plt.xlabel('年份')
plt.ylabel('平均词频')
plt.grid(True, axis='y', linestyle='--', alpha=0.7)
plt.xticks(range(2016, 2024))
plt.legend()
plt.tight_layout()
plt.show()

<Figure size 720x432 with 1 Axes>